In [2]:
import math
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device.type)

cuda


Defining the function space

In [3]:
def sample_poly_coeffs(batch_size : int, degree : int, scale : float):

    return scale * torch.rand(batch_size, degree+1, device=device)

In [60]:
#True values for the polynomial
def eval_poly(coeffs : torch.tensor, x : torch.tensor):
    """
    coeffs : (B, D+1)
    x : (B, N, 1)
    return f : (B, N, 1)
    """

    B, D1 = coeffs.shape
    D = D1 - 1
    '''
    x = x.to(device)
    coeffs = coeffs.to(device)
    '''

    if x.dim() == 2:
        # converting (N, 1) to (B, N, 1) if applicable
        x = x.unsqueeze(0).repeat(B, 1, 1) 
        """
        unsqueeze expands the tensor by adding a dimension at a specified position.
        repeat acts like numpy tile.
        """

    powers = torch.cat([x**k for k in range(D+1)], dim=-1)

    #coeffs is in the shape(B+1, D+1)
    c = coeffs.unsqueeze(1)  #inserting another dim at index=1
    f = (powers * c).sum(dim=-1, keepdim=True)
    return f


DEFINING THE MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_layers, out_dim, in_dim, act=nn.Tanh):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_layers:
            layers += [nn.Linear(prev, h), act()]
            prev = h

        layers += [nn.Linear(prev, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class DeepONet(nn.Module):
    def __init__(self, m_sensors : int, p : int):
        super().__init__()
        '''
        setting up two networks of two layers for trunk and branch
        '''

        self.branch = MLP(in_dim = m_sensors, hidden_layers=[64, 64], out_dim=p, act=nn.Tanh)

        self.trunk = MLP(in_dim=1, hidden_layers=[64, 64], out_dim=p, act=nn.Tanh)


    def forward(self, f_sensors, x):
        B = f_sensors.shape[0]

        if x.dim() == 2:
            x_in = x.unsqueeze(0).repeat(B, 1, 1)
        else:
            x_in = x


        b = self.branch(f_sensors) #function inputs for the latent vector

        t = self.trunk(x_in) # spatial points for the other latent vector

        u = (t * b.unsqueeze(1)).sum(dim=-1, keepdim=True)

        u = x_in * (1.0 - x_in) * u #hard boundary condition(at x = 0, u= 0 and at x = 1, u = 0)

        return u

Computing Second spatial derivatives(for the governing equation)

In [48]:
def second_derivatives(u, x):

    du_dx = torch.autograd.grad(
        outputs = u,
        inputs = x,
        grad_outputs=torch.ones_like(u),
        create_graph = True,
        retain_graph=True
    )[0]

    d2u_dx2 = torch.autograd.grad(
        outputs = du_dx,
        inputs = x,
        grad_outputs=torch.ones_like(du_dx),
        create_graph=True,
        retain_graph=True
    )[0]

    return d2u_dx2


In [49]:
torch.manual_seed(0)
np.random.seed(0)

m = 10
p = 32
N_domain = 100
batch_size = 64

x_sensors = torch.linspace(0, 1, m).view(-1, 1)

x_domain = torch.linspace(0, 1, N_domain).view(-1 ,1)

model = DeepONet(m_sensors=m, p=p).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

w_pde = 1.0
w_bc = 10.0

In [50]:
def train_step():
    model.train()
    optimizer.zero_grad()

    coeffs = sample_poly_coeffs(batch_size=batch_size, degree=3, scale=0.1)

    f_sensor_vals = eval_poly(coeffs, x_sensors).squeeze(-1)

    f_domain_vals = eval_poly(coeffs, x_domain)

    x_in = x_domain.unsqueeze(0).repeat(batch_size, 1, 1).clone().detach().to(device).requires_grad_(True)

    u = model(f_sensor_vals, x_in)

    u_xx = second_derivatives(u, x_in)

    res = -u_xx - f_domain_vals

    loss_pde = (res**2).mean()

    x0 = torch.zeros(batch_size, 1, 1, device=device, requires_grad=False)
    x1 = torch.ones(batch_size, 1, 1, device=device, requires_grad=False)

    u0 = model(f_sensor_vals, x0)
    u1 = model(f_sensor_vals, x1)

    loss_bc = (u0**2).mean() + (u1**2).mean()

    loss = w_pde * loss_pde + w_bc * loss_bc
    loss.backward()
    optimizer.step()

    return loss.item(), loss_pde.item(), loss_bc.item()

    




TRAINING LOOOP

In [51]:
steps = 5000
print_every = 500

for step in range(1, steps+1):
    loss, loss_pde, loss_bc = train_step()
    if step % print_every == 0 or step == 1:
        print(f"#-------Step: {step}--------#")
        print(f"Total Loss: {loss:.3e}")
        print(f"PDE Loss: {loss_pde:.3e}")
        print(f"BC Loss: {loss_bc:.3e}")
        print("#----------------------------#")


#-------Step: 1--------#
Total Loss: 4.699e-03
PDE Loss: 4.699e-03
BC Loss: 0.000e+00
#----------------------------#
#-------Step: 500--------#
Total Loss: 7.688e-06
PDE Loss: 7.688e-06
BC Loss: 0.000e+00
#----------------------------#
#-------Step: 1000--------#
Total Loss: 6.173e-06
PDE Loss: 6.173e-06
BC Loss: 0.000e+00
#----------------------------#
#-------Step: 1500--------#
Total Loss: 5.898e-06
PDE Loss: 5.898e-06
BC Loss: 0.000e+00
#----------------------------#
#-------Step: 2000--------#
Total Loss: 4.470e-06
PDE Loss: 4.470e-06
BC Loss: 0.000e+00
#----------------------------#
#-------Step: 2500--------#
Total Loss: 2.264e-06
PDE Loss: 2.264e-06
BC Loss: 0.000e+00
#----------------------------#
#-------Step: 3000--------#
Total Loss: 8.828e-07
PDE Loss: 8.828e-07
BC Loss: 0.000e+00
#----------------------------#
#-------Step: 3500--------#
Total Loss: 4.599e-07
PDE Loss: 4.599e-07
BC Loss: 0.000e+00
#----------------------------#
#-------Step: 4000--------#
Total Loss: 2.69

MODEL EVALUATION

In [61]:
model.eval()
n_plots = 3

x_plot = torch.linspace(0, 1, 200, device=device).view(-1, 1)

coeffs = sample_poly_coeffs(batch_size=n_plots, degree = 3, scale=0.1)
f_sensor_vals = eval_poly(coeffs, x_sensors).squeeze(-1)

with torch.no_grad():
    u_pred = model(f_sensor_vals, x_plot)

u_pred = u_pred.squeeze(-1).cpu().numpy()
f_sensor = eval_poly(coeffs, x_plot).squeeze(-1).cpu().numpy()

x_plot_np = x_plot.squeeze(-1).cpu().numpy()
x_sensors_np = x_sensors.squeeze(-1).cpu().numpy()
f_sensor_np = eval_poly(coeffs, x_sensors_np).squeeze(-1).cpu().numpy()


plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("DeepONet eval")
plt.axhline(0, linewidth=1, alpha=0.2)

for i in range(n_plots):
    plt.plot(x_plot_np, u_pred[i], linewidth=2)

plt.tight_layout()
plt.show()





RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!